# Lab 04 – Decision Tree
## Lab Task 1: Predict Titanic Survival

**Goal:** Build a decision tree model that predicts whether a passenger survived the Titanic, using parameters such as class, sex, age and fare, and calculate the score of the model.

**Before you start:** download `train.csv` from https://www.kaggle.com/c/titanic/data (free Kaggle account needed). Then run the cells below from top to bottom.

## Step 1: Import libraries

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
import matplotlib.pyplot as plt

## Step 2: Load the dataset
Run the cell and choose `train.csv` when the file picker appears.

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv("train.csv")
df.head()

## Step 3: Explore the data
Check the size, column types and missing values.

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
# missing values per column
df.isnull().sum()

`Age` (177), `Cabin` (687) and `Embarked` (2) have missing values.

## Step 4: Clean the data
- Drop columns that do not help prediction: `PassengerId`, `Name`, `Ticket` (unique identifiers) and `Cabin` (mostly empty).
- Fill missing `Age` with the median and missing `Embarked` with the most common value.

In [ ]:
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis='columns')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df.isnull().sum()

## Step 5: Encode text columns
Machine learning algorithms only work with numbers, so `Sex` and `Embarked` are converted with `LabelEncoder` (same approach as Activity 2).

In [ ]:
le_sex = LabelEncoder()
le_embarked = LabelEncoder()

df['Sex'] = le_sex.fit_transform(df['Sex'])
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])

print("Sex mapping:     ", dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_))))
print("Embarked mapping:", dict(zip(le_embarked.classes_, le_embarked.transform(le_embarked.classes_))))
df.head()

## Step 6: Separate features and target, then split into train and test sets
- **Target (y):** `Survived` (0 = died, 1 = survived)
- **Features (X):** all other columns
- 70% of the data is used for training and 30% for testing.

In [ ]:
X = df.drop('Survived', axis='columns')
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1)

print("Training samples:", X_train.shape[0])
print("Testing samples: ", X_test.shape[0])

## Step 7: Build and train the Decision Tree model

In [ ]:
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

## Step 8: Calculate the score of the model

In [ ]:
train_score = model.score(X_train, y_train)
y_pred = model.predict(X_test)
test_accuracy = metrics.accuracy_score(y_test, y_pred)

print("Training score:", train_score)
print("Test accuracy: ", test_accuracy)

**Observation:** the training score is very high (close to 1.0) while the test accuracy is lower. The unpruned tree has memorized the training data, which is **overfitting**.

In [ ]:
print("Confusion matrix:")
print(metrics.confusion_matrix(y_test, y_pred))
print()
print(metrics.classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

## Step 9: Improve the model with pruning
Limiting `max_depth` (pre-pruning) stops the tree from growing too complex. Here we try several depths with the `entropy` criterion and compare test accuracy.

In [ ]:
results = []
for depth in range(1, 11):
    clf_try = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=1)
    clf_try.fit(X_train, y_train)
    results.append((depth,
                    clf_try.score(X_train, y_train),
                    metrics.accuracy_score(y_test, clf_try.predict(X_test))))

results_df = pd.DataFrame(results, columns=['max_depth', 'train_score', 'test_accuracy'])
results_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(results_df['max_depth'], results_df['train_score'], marker='o', label='Training score')
plt.plot(results_df['max_depth'], results_df['test_accuracy'], marker='o', label='Test accuracy')
plt.xlabel('max_depth')
plt.ylabel('Score')
plt.title('Effect of max_depth on accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Step 10: Final pruned model
Using `max_depth=3` for a simple, easy to explain tree. Change this number if the table above shows a better depth.

In [ ]:
clf = DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Pruned model training score:", clf.score(X_train, y_train))
print("Pruned model test accuracy: ", metrics.accuracy_score(y_test, y_pred))

## Step 11: Visualize the decision tree

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(clf,
          feature_names=X.columns,
          class_names=['Died', 'Survived'],
          filled=True, rounded=True, fontsize=10)
plt.show()

## Step 12: Which features matter most?

In [ ]:
importance = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance)

importance.plot(kind='bar', figsize=(7, 4), title='Feature importance')
plt.show()

## Step 13: Predict survival for new passengers
Feature order: `Pclass, Sex, Age, SibSp, Parch, Fare, Embarked`
(Sex: female=0, male=1. Embarked: C=0, Q=1, S=2.)

In [ ]:
# Passenger 1: 3rd class man, age 22, travelling with 1 sibling/spouse, low fare, embarked at Southampton
p1 = pd.DataFrame([[3, 1, 22, 1, 0, 7.25, 2]], columns=X.columns)

# Passenger 2: 1st class woman, age 38, high fare, embarked at Cherbourg
p2 = pd.DataFrame([[1, 0, 38, 1, 0, 71.28, 0]], columns=X.columns)

for name, p in [("Passenger 1 (3rd class man, 22)", p1), ("Passenger 2 (1st class woman, 38)", p2)]:
    result = clf.predict(p)[0]
    print(name, "->", "Survived" if result == 1 else "Did not survive")

## Conclusion
- A decision tree was built on the Titanic dataset to predict survival.
- The unpruned tree overfits (very high training score, lower test accuracy).
- Pruning with `max_depth` and the `entropy` criterion gives a simpler model that generalizes better.
- `Sex`, `Pclass`/`Fare` and `Age` are the most influential features.